In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import spacy
from textblob import TextBlob
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import faiss

c:\Users\Filipe\Documents\Estudos e testes\Desafio A3\books_data_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Data overview

In [3]:
df_books = pd.read_csv("../data/raw/books_data.csv", nrows=100000)
df_books.head(2)

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...,NaN,1996,http://books.google.nl/books?id=DykPAAAACAAJ&d...,['Comics & Graphic Novels'],NaN
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN


In [4]:
df_reviews = pd.read_csv("../data/raw/Books_rating.csv", nrows=100000)
# df_reviews = df_reviews.sample(frac=0.001)
df_reviews.head(5)

,Id,Title,Price,User_id,profileName,score,time,summary,text
0,1882931173,Its Only Art If Its Well Hung!,NaN,AVCGYZL8FQQTD,"Jim of Oz ""jim-of-oz""",4.0,940636800,Nice collection of Julie Strain images,This is only for Julie Strain fans. It's a col...
1,0826414346,Dr. Seuss: American Icon,NaN,A30TK6U7DNS82R,Kevin Killian,5.0,1095724800,Really Enjoyed It,I don't care much for Dr. Seuss but after read...
2,0826414346,Dr. Seuss: American Icon,NaN,A3UH4UZ4RSVO82,John Granger,5.0,1078790400,Essential for every personal and Public Library,"If people become the books they read and if ""t..."
3,0826414346,Dr. Seuss: American Icon,NaN,A2MVUWT453QH61,"Roy E. Perry ""amateur philosopher""",4.0,1090713600,Phlip Nel gives silly Seuss a serious treatment,"Theodore Seuss Geisel (1904-1991), aka &quot;D..."
4,0826414346,Dr. Seuss: American Icon,NaN,A22X4XUPKF66MR,"D. H. Richards ""ninthwavestore""",4.0,1107993600,Good academic overview,Philip Nel - Dr. Seuss: American IconThis is b...


In [5]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Id           100000 non-null  object 
 1   Title        99999 non-null   object 
 2   Price        12982 non-null   float64
 3   User_id      80784 non-null   object 
 4   profileName  80780 non-null   object 
 5   score        100000 non-null  float64
 6   time         100000 non-null  int64  
 7   summary      99982 non-null   object 
 8   text         100000 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 6.9+ MB


In [6]:
df_merged = df_reviews.merge(df_books[["Title", "authors", "publishedDate", "categories"]], on="Title", how="left")
df_merged.columns

Index(['Id', 'Title', 'Price', 'User_id', 'profileName', 'score', 'time',
       'summary', 'text', 'authors', 'publishedDate', 'categories'],
      dtype='object')

#### General analysis

In [7]:
# avg score per book

# df_reviews.groupby("Title")["score"].mean().sort_values(ascending=False).head(10)
df_reviews.groupby("Title").agg(
    avg_score=("score", "mean"),
    n_reviews=("score", "count")
).sort_values(
    by=["avg_score", "n_reviews"],
    ascending=[False, False]
).head(10)

,avg_score,n_reviews
Title,,
Attaining the Worlds Beyond: A Guide to Spiritual Discovery,5.0,25
Children of the Star,5.0,22
Walking in Victory: Experiencing True Sanctification and Holiness through God's Grace,5.0,20
Rapid Descent : Disaster in Boston Harbor,5.0,19
"HARDTACK AND COFFEE, OR THE UNWRITTEN STORY OF ARMY LIFE",5.0,18
"Going for the Bronze: Still Bitter, More Baggage",5.0,17
Babyface: A Story of Heart and Bones,5.0,16
Natural Theology ; Evidences of the Existence and Attributes of the Deity. Collected from the Appearances of Nature.,5.0,16
"Sam Shepard, Seven Plays",5.0,15


In [8]:
# avg score per author

df_merged.groupby("authors").agg(
    avg_score=("score", "mean"),
    n_reviews=("score", "count"),
    n_books=("Title", "nunique")
).sort_values(
    by=["avg_score", "n_reviews", "n_books"],
    ascending=[False, False, False]
).head(10)

,avg_score,n_reviews,n_books
authors,,,
['Michael Laitman'],5.0,25,1
['J. P. Polidoro'],5.0,22,2
['Sylvia Engdahl'],5.0,22,1
['Charles Trumbull'],5.0,20,1
['John D. Billings'],5.0,18,1
"['Sloane Tanen', 'Stefan Hagen']",5.0,17,1
['Jeanne McDermott'],5.0,16,1
['William Paley'],5.0,16,1
['Michel Gagne'],5.0,15,2


In [9]:
# avg score per genre

df_merged.groupby("categories").agg(
    avg_score=("score", "mean"),
    n_reviews=("score", "count"),
    n_books=("Title", "nunique")
).sort_values(
    by=["avg_score", "n_reviews", "n_books"],
    ascending=[False, False, False]
).head(10)

,avg_score,n_reviews,n_books
categories,,,
['Natural history'],5.0,19,3
['Brazil'],5.0,10,1
['Fund raising'],5.0,10,1
['Babies'],5.0,9,1
['Dee Jen-Djieh (Fictitious character)'],5.0,9,1
"[""'Abd al-Bah̄a, 1844-1921""]",5.0,8,1
['Contaminated sediments'],5.0,8,1
['Czechoslovakia'],5.0,7,1
['Death'],5.0,6,2


In [10]:
# books with the most reviews

df_reviews.groupby("Title").agg(
    n_reviews=("score", "count"),
    avg_score=("score", "mean")
).sort_values("n_reviews", ascending=False).head(10)

,n_reviews,avg_score
Title,,
"The Hobbitt, or there and back again; illustrated by the author.",4420,4.658597
Fahrenheit 451,3287,4.158199
"ERAGON: INHERITANCE, BOOK ONE.",3137,3.834874
1984,1895,4.563588
Lord of the flies,1556,3.921594
Dune,1522,4.420499
Of Mice and Men (Penguin Audiobooks),1340,4.350746
Foundation,1214,4.275124
The Other Boleyn Girl,1074,4.285847


In [11]:
# users with most reviews

df_reviews.groupby("profileName").agg(
    n_reviews=("score", "count"),
    avg_score=("score", "mean")
).sort_values("n_reviews", ascending=False).head(10)

,n_reviews,avg_score
profileName,,
A Customer,200,4.115000
Midwest Book Review,182,4.994505
"E. A Solinas ""ea_solinas""",112,4.687500
Harriet Klausner,93,4.634409
"Shalom Freedman ""Shalom Freedman""",65,4.707692
"Blue Tyson ""- Research Finished""",63,3.825397
John,62,4.193548
Avid Reader,61,3.901639
"bernie ""xyzzy""",58,4.931034


In [12]:
# correlation between price, score and number of reviews

df_comp = df_merged[['Price', 'score']]
df_comp['n_reviews'] = df_merged.groupby("Title")["score"].transform("count")

df_comp[['Price', 'score', 'n_reviews']].corr()

C:\Users\Filipe\AppData\Local\Temp\ipykernel_12372\4289811522.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_comp['n_reviews'] = df_merged.groupby("Title")["score"].transform("count")


,Price,score,n_reviews
Price,1.000000,0.000674,-0.060944
score,0.000674,1.000000,0.036186
n_reviews,-0.060944,0.036186,1.000000


#### NLP Preprocesssing

In [157]:
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    doc = nlp(text.lower())
    tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
    return " ".join(tokens)

df_reviews["clean_review"] = df_reviews["text"].apply(clean_text)

In [158]:
df_reviews.sample(10)[["text", "clean_review"]]

,text,clean_review
2310214,"This is a brilliant, heart-felt, humorous yet ...",brilliant heart feel humorous introduction soc...
2538032,1000+ pages is very intimidating. I know; some...,page intimidating know suggest thought read fo...
2381087,I love this story; I hate those times when peo...,love story hate time people judgemental guess ...
1672548,Brock's supposed 'turnaround' is questionable ...,brock suppose turnaround questionable well lat...
2485010,"Ho hum, still waiting for a publisher rewrite ...",ho hum wait publisher rewrite poor kindle vers...
1654124,Williams is the father of the revisionist myth...,williams father revisionist myth go wrong worl...
718583,This books is not helpful if you are an impend...,book helpful impending mother want balanced in...
704800,"From the Publisher:""Alison Dundes Renteln's ""T...",dunde renteln cultural defense extraordinarily...
741069,"After two false starts, Robin Pilcher has come...",false start robin pilcher come new novel risk ...
1311433,I am slowly working my way through his bibliog...,slowly work way bibliography up down pratchett...


#### Sentiment Analysis

In [159]:
df_reviews["sentiment"] = df_reviews["clean_review"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

In [160]:
# books with highest average sentiment

title_sentiment = df_reviews.groupby("Title")["sentiment"].mean()
title_sentiment.sort_values(ascending=False).head(10)

Title
Research Strategies: Finding Your Way Through the Information Fog                       1.000000
Internet Performance Survival Guide: QoS Strategies for Multiservice Networks           1.000000
Favoured Child (Wideacre Trilogy 2)                                                     1.000000
Prince of Dreams: A Tale of Tristan and Essylte                                         1.000000
Exodus from Obesity: 2nd Edition                                                        1.000000
My Friends' Secrets:Conversations with My Friends about Beauty, Health and Happiness    1.000000
Disposable: A History of Skateboard Art                                                 1.000000
The Tortilla Curtain                                                                    1.000000
The Hart Brothers Simon & Callaghan: Beloved\Callaghan's Bride                          0.933333
Joy in Our Weakness: A Gift of Hope from the Book of Revelation                         0.933333
Name: sentiment, dtype: 

In [15]:
# best scored authors

# df_merged_2 = df_reviews.merge(df_books[["Title", "authors", "categories"]], on="Title", how="left")
df_merged_2 = pd.read_csv("../data/processed/processed_reviews.csv", nrows=100000)
author_sentiment = df_merged_2.groupby("authors")["sentiment"].mean()
author_sentiment.sort_values(ascending=False).head(10)

authors
Jim Jonson                                                  5.0000
John Stewart Davenport                                      5.0000
Percy George Osborn                                         5.0000
M. S. Qawwee                                                5.0000
William H. Roach, Aspen Health Law and Compliance Center    5.0000
Brian H. Jones                                              5.0000
Elizabeth Bowen                                             4.8125
Ellen Pritchard Dodge                                       4.7500
Kazuo Iwamura                                               4.7500
D. A. Cruse                                                 4.7500
Name: sentiment, dtype: float64

In [18]:
# best scored authors

df_authors = df_merged_2.groupby("authors").agg(
    avg_score=("score", "mean"),
    avg_sentiment=("sentiment", "mean"),
    n_reviews=("score", "count")
)

df_authors[df_authors["n_reviews"] > 10].sort_values(
    by=["avg_sentiment", "avg_score", "n_reviews"],
    ascending=[False, False, False]
).head(10)

,avg_score,avg_sentiment,n_reviews
authors,,,
Gloria Houston,4.882353,3.756635,17
Max Lucado,4.857143,3.633029,28
"Various Authors,",4.842105,3.625776,19
"Rachael Hale, Suzanne McFadden",4.833333,3.613582,12
"Mark A. Vieira, George Hurrell",4.941176,3.602349,17
Charles Lindsay,4.604167,3.594665,48
Rebecca Harvin,4.769231,3.557372,13
"Katharine Lee Bates, Scholastic Inc.",4.882353,3.550978,17
"Terry Wilson, Roxanne Wilson",4.588235,3.539698,17


In [22]:
# best scored genres

df_genres = df_merged_2.groupby("categories").agg(
    avg_score=("score", "mean"),
    avg_sentiment=("sentiment", "mean"),
    n_reviews=("score", "count")
)

df_genres[df_genres["n_reviews"] > 50].sort_values(
    by=["avg_sentiment", "avg_score", "n_reviews"],
    ascending=[False, False, False]
).head(10)

,avg_score,avg_sentiment,n_reviews
categories,,,
Baking,4.909871,3.389972,233
"Education, Higher",4.193548,3.242890,62
Cooking,4.315217,3.203299,644
Crafts & Hobbies,4.293423,3.174868,593
Antiques & Collectibles,4.470270,3.171406,185
Families,4.055556,3.166508,54
Travel,4.343750,3.157168,224
Netherlands,4.736111,3.153608,72
Pets,3.974026,3.132266,154


In [ ]:
# best scored books

df_books = df_merged_2.groupby("Title").agg(
    avg_score=("score", "mean"),
    avg_sentiment=("sentiment", "mean"),
    n_reviews=("score", "count")
)

df_books[df_books["n_reviews"] > 10].sort_values(
    by=["avg_sentiment", "avg_score", "n_reviews"],
    ascending=[False, False, False]
).head(10)

,avg_score,avg_sentiment,n_reviews
Title,,,
The Song of Hiawatha,5.0,3.505417,12
Babyface: A Story of Heart and Bones,5.0,3.410227,16
"Cooking with Herb, the Vegetarian Dragon: A Cookbook for Kids",5.0,3.299511,11
"Sam Shepard, Seven Plays",5.0,3.185973,15
She'll Be Comin' 'Round the Mountain,5.0,3.180162,11
Rapid Descent : Disaster in Boston Harbor,5.0,3.178464,19
Welding with Children: Stories,5.0,3.146218,12
Is Jesus the Only Savior?,5.0,3.109680,12
Children of the Star,5.0,3.088856,22


#### Semantic search

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

df_reviews["embedding"] = df_reviews["clean_review"].apply(
    lambda x: model.encode(x)
)

embeddings = np.vstack(df_reviews["embedding"].values)

index = faiss.IndexFlatL2(384)
index.add(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6552.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
def search_reviews(query, top_k=10):
    query_vector = model.encode([query])
    D, I = index.search(query_vector, top_k)
    return D, I

In [60]:
# search_query = "I love this book"
search_query = "reviews that deeply analyze character development"

In [98]:
D, I = search_reviews(search_query, top_k=5)
for distance, idx in zip(D[0], I[0]):
    print("Review relevance:", distance)
    print("Review:", df_reviews.iloc[idx]["text"])
    print("User:", df_reviews.iloc[idx]["profileName"])
    print()

Review relevance: 1.0500581
Review: When I first read this the I was mezmerized at the style of writting. I soon became so hooked on this book that I wanted to fall on my knees and shut out the rest of the world. Haddon delivers a portrayal of a woman that is hurt loney and abused and I think she does a wonderful job making the reader feel like the character.
User: Clara

Review relevance: 1.0920141
Review: It's glaringly obvious that all of the glowing reviews have been written by the same person, perhaps the author herself. They all have the same misspellings and poor sentence structure that is featured in the book. Who made Veronica Haddon think she is an author?
User: Bronx Girl

Review relevance: 1.1323907
Review: I purchased this book after reading rave reviews of it included within a review of a really good book I'd read. Apparently, the author (or her friends and family) are placing reviews of good books and encouraging people to buy this book if you liked the one they were rev

#### Users clustering

In [107]:
model = SentenceTransformer('all-MiniLM-L6-v2')

df_reviews["embedding"] = df_reviews["clean_review"].apply(
    lambda x: model.encode(x)
)

embeddings = np.vstack(df_reviews["embedding"].values)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7339.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [108]:
kmeans = KMeans(n_clusters=8)
df_reviews["cluster"] = kmeans.fit_predict(embeddings)

In [111]:
df_reviews[['profileName','cluster']].sort_values("cluster")

,profileName,cluster
96,"Kiki B. ""The Shepherdess""",0
97,Heddlemaid,0
98,"Linda Pool ""cookbookaholic""",0
92,Gary W. Marian,0
91,NaN,0
...,...,...
11,Dr. Terry W. Dorsett,7
56,"Nov 10 ""Tom""",7
49,GodsBreath.wordpress,7
48,haskell,7
